# EDA longitudinal ENIGH 2018-2024

Este notebook conserva los displays de las bases originales y de las tablas homologadas/apiladas. Después de esos displays continúo con el EDA desde variables numéricas en adelante; la revisión de estabilidad quedó separada en `estabilidad_de_bases.ipynb`.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 90)
pd.set_option("display.max_rows", 90)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

RAW_ROOT = ROOT / "data/raw/EINGH"
INTERIM_ROOT = ROOT / "data/interim/revision_1"
YEARS = [2018, 2020, 2022, 2024]
CORE_TABLES = ["concentradohogar", "hogares", "ingresos", "poblacion", "trabajos", "viviendas"]
STACK_PATHS = {name: INTERIM_ROOT / f"{name}_common_2018_2024.csv.gz" for name in CORE_TABLES}
METADATA = pd.read_csv(ROOT / "docs/enigh_variable_metadata.csv", dtype=str)

## Bases originales

Primero cargo y muestro las primeras 20 filas de las bases originales usadas para construir las tablas homologadas.

In [ ]:
original_samples = {}
for table in CORE_TABLES:
    for year in YEARS:
        df = pd.read_csv(RAW_ROOT / str(year) / f"{table}.csv", nrows=20, low_memory=False, encoding="utf-8-sig")
        original_samples[(table, year)] = df
        print(f"{table} {year}")
        display(df.head(20))

## Tablas homologadas y apiladas

Después cargo las seis tablas finales 2018-2024 y muestro sus primeras 20 filas.

In [ ]:
tables = {}
for table in CORE_TABLES:
    df = pd.read_csv(STACK_PATHS[table], low_memory=False)
    tables[table] = df
    globals()[table] = df
    print(f"{table} homologada/apilada 2018-2024")
    display(df.head(20))

## Preparación mínima para el EDA

Reconstruyo únicamente los objetos necesarios para los análisis posteriores. La estabilidad y las exclusiones se revisan en el notebook separado.

In [ ]:
def clean_for_eda(df):
    clean = df.copy()
    for col in clean.columns:
        if pd.api.types.is_object_dtype(clean[col]) or str(clean[col].dtype).startswith("string"):
            clean[col] = clean[col].astype("string").str.strip()
            clean[col] = clean[col].replace({"": pd.NA, "NA": pd.NA, "N/A": pd.NA})
    return clean

clean_tables = {table: clean_for_eda(df) for table, df in tables.items()}

## 5. Variables numéricas

In [ ]:
NUMERIC_RELEVANT = {
    "concentradohogar": ["ing_cor", "ingtrab", "trabajo", "sueldos", "negocio", "hospital", "gasto_mon", "tot_integ", "ocupados", "percep_ing", "edad_jefe"],
    "ingresos": ["ing_tri"],
    "poblacion": ["edad", "grado", "gradoaprob", "num_trabaj", "hijos_viv", "hijos_mue", "hijos_sob"],
    "trabajos": ["htrab"],
    "hogares": ["num_carret", "num_pickup", "num_auto", "num_compu"],
    "viviendas": ["num_cuarto", "cuart_dorm", "tot_resid", "renta"],
}

def numeric_stats(df, cols, by_year=False):
    rows = []
    for col in [c for c in cols if c in df.columns]:
        groups = df.groupby("anio") if by_year else [(None, df)]
        for year, g in groups:
            s = pd.to_numeric(g[col], errors="coerce")
            rows.append({
                "anio": year,
                "variable": col,
                "count": s.notna().sum(),
                "mean": s.mean(),
                "median": s.median(),
                "std": s.std(),
                "p25": s.quantile(.25),
                "p75": s.quantile(.75),
                "p95": s.quantile(.95),
                "p99": s.quantile(.99),
                "min": s.min(),
                "max": s.max(),
                "zero_pct": (s == 0).mean() * 100,
                "negative_pct": (s < 0).mean() * 100,
            })
    return pd.DataFrame(rows)

numeric_stats_all = {table: numeric_stats(clean_tables[table], cols) for table, cols in NUMERIC_RELEVANT.items()}
numeric_stats_year = {table: numeric_stats(clean_tables[table], cols, by_year=True) for table, cols in NUMERIC_RELEVANT.items()}

for table, stats in numeric_stats_all.items():
    print("\n" + "=" * 90)
    print(table)
    display(stats.round(2))

In [ ]:
for table, cols in NUMERIC_RELEVANT.items():
    df = clean_tables[table]
    for col in [c for c in cols if c in df.columns][:6]:
        s = pd.to_numeric(df[col], errors="coerce")
        if s.notna().sum() < 100:
            continue
        upper = s.quantile(.99)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(s[s <= upper], bins=45, kde=True, ax=axes[0], color="#4C78A8")
        axes[0].set_title(f"{table}.{col}: histograma hasta P99")
        sns.boxplot(data=df.assign(_v=s.clip(upper=upper)), x="anio", y="_v", showfliers=False, ax=axes[1], color="#A0CBE8")
        axes[1].set_title(f"{table}.{col}: boxplot por año")
        axes[1].set_xlabel("Año")
        axes[1].set_ylabel(col)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

In [ ]:
for table, stats in numeric_stats_year.items():
    keep = stats[stats["variable"].isin(NUMERIC_RELEVANT[table][:6])]
    if keep.empty:
        continue
    display(keep.round(2).head(60))
    for col in keep["variable"].unique()[:4]:
        temp = keep[keep["variable"] == col]
        fig, ax = plt.subplots(figsize=(7, 3.5))
        sns.lineplot(data=temp, x="anio", y="median", marker="o", label="mediana", ax=ax)
        sns.lineplot(data=temp, x="anio", y="mean", marker="o", label="media", ax=ax)
        ax.set_title(f"{table}.{col}: centro por año")
        ax.set_xticks(YEARS)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

## 6. Variables categóricas

In [ ]:
CATEGORICAL_RELEVANT = {
    "concentradohogar": ["tam_loc", "est_socio", "clase_hog", "sexo_jefe", "educa_jefe"],
    "hogares": ["telefono", "celular", "conex_inte", "tarjeta"],
    "poblacion": ["sexo", "nivelaprob", "edo_conyug", "trabajo_mp"],
    "trabajos": ["subor", "indep", "contrato", "tiene_suel", "tipoact"],
    "viviendas": ["tipo_viv", "combustible", "medidor_luz", "tenencia", "drenaje", "disp_elect"],
}

cat_summary_rows = []
cat_year_tables = {}
for table, cols in CATEGORICAL_RELEVANT.items():
    df = clean_tables[table]
    for col in [c for c in cols if c in df.columns]:
        s = df[col].astype("string")
        vc = s.value_counts(dropna=False, normalize=True)
        cat_summary_rows.append({
            "tabla": table,
            "variable": col,
            "categorias": s.nunique(dropna=True),
            "dominante": vc.index[0],
            "dominante_pct": vc.iloc[0] * 100,
            "missing_pct": s.isna().mean() * 100,
            "categorias_menor_1pct": int((s.value_counts(normalize=True) < .01).sum()),
        })
        tab = pd.crosstab(df["anio"], s, normalize="index") * 100
        cat_year_tables[(table, col)] = tab

cat_summary = pd.DataFrame(cat_summary_rows)
cat_summary.round(2)

In [ ]:
for (table, col), tab in list(cat_year_tables.items()):
    top = tab.mean().sort_values(ascending=False).head(8).index
    display(tab[top].round(2))
    plot_df = tab[top].reset_index().melt(id_vars="anio", var_name=col, value_name="pct")
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(data=plot_df, x="anio", y="pct", hue=col, ax=ax)
    ax.set_title(f"{table}.{col}: composición por año")
    ax.set_xlabel("Año")
    ax.set_ylabel("%")
    ax.legend(title=col, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 7. Ingresos

In [ ]:
income_meta = METADATA[
    (METADATA["table"].isin(["concentradohogar.csv", "ingresos.csv"]))
    & (METADATA["variable"].isin(["ingtrab", "ing_cor", "trabajo", "sueldos", "negocio", "ing_tri"]))
][["year", "table", "variable", "label", "dtype"]].drop_duplicates()
income_meta.sort_values(["table", "variable", "year"])

In [ ]:
income_df = clean_tables["concentradohogar"].copy()
income_df["ingtrab"] = pd.to_numeric(income_df["ingtrab"], errors="coerce")
income_df["log1p_ingtrab"] = np.log1p(income_df["ingtrab"].clip(lower=0))
income_p99 = income_df["ingtrab"].quantile(.99)

income_summary = numeric_stats(income_df, ["ingtrab"])
income_year = numeric_stats(income_df, ["ingtrab"], by_year=True)
display(income_summary.round(2))
display(income_year.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(income_df.loc[income_df["ingtrab"] <= income_p99, "ingtrab"], bins=60, kde=True, ax=axes[0], color="#4C78A8")
axes[0].set_title("ingtrab hasta P99")
sns.histplot(income_df["log1p_ingtrab"], bins=60, kde=True, ax=axes[1], color="#59A14F")
axes[1].set_title("log1p(ingtrab)")
sns.boxplot(data=income_df.assign(_v=income_df["ingtrab"].clip(upper=income_p99)), x="anio", y="_v", showfliers=False, ax=axes[2], color="#A0CBE8")
axes[2].set_title("ingtrab por año")
axes[2].set_ylabel("ingtrab")
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for col in ["median", "p25", "p75", "p95"]:
    sns.lineplot(data=income_year, x="anio", y=col, marker="o", label=col, ax=ax)
ax.set_title("Evolución nominal de ingtrab")
ax.set_xticks(YEARS)
ax.set_xlabel("Año")
ax.set_ylabel("ingtrab")
plt.tight_layout()
plt.show()
plt.close(fig)

## 8. Variables numéricas vs ingresos

In [ ]:
income_target = income_df["ingtrab"]
numeric_income_rows = []
for col in [c for c in clean_tables["concentradohogar"].columns if c not in ["anio", "folioviv", "foliohog", "ingtrab"]]:
    x = pd.to_numeric(clean_tables["concentradohogar"][col], errors="coerce")
    if x.notna().mean() < .80 or x.nunique(dropna=True) <= 2:
        continue
    valid = x.notna() & income_target.notna()
    xv, yv = x[valid], income_target[valid]
    numeric_income_rows.append({
        "variable": col,
        "Pearson": xv.corr(yv),
        "Spearman": xv.rank().corr(yv.rank()),
        "n": int(valid.sum()),
    })
income_correlations = pd.DataFrame(numeric_income_rows)
income_correlations["abs_spearman"] = income_correlations["Spearman"].abs()
income_correlations = income_correlations.sort_values("abs_spearman", ascending=False)
income_correlations.head(30).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=income_correlations.head(20), y="variable", x="Spearman", color="#F28E2B", ax=ax)
ax.axvline(0, color="black", linewidth=.8)
ax.set_title("Correlación Spearman con ingtrab")
ax.set_xlabel("Spearman")
ax.set_ylabel("")
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
top_scatter = income_correlations.head(4)["variable"].tolist()
scatter = income_df[["ingtrab"] + top_scatter].apply(pd.to_numeric, errors="coerce").dropna().sample(n=5000, random_state=42)
scatter["log1p_ingtrab"] = np.log1p(scatter["ingtrab"].clip(lower=0))
for col in top_scatter:
    tmp = scatter[[col, "log1p_ingtrab"]].dropna().copy()
    tmp["bin"] = pd.qcut(tmp[col].rank(method="first"), q=20, duplicates="drop")
    binned = tmp.groupby("bin", observed=True).agg(x_med=(col, "median"), y_med=("log1p_ingtrab", "median")).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(7, 4.2))
    sns.scatterplot(data=tmp, x=col, y="log1p_ingtrab", alpha=.15, s=8, edgecolor=None, ax=ax)
    sns.lineplot(data=binned, x="x_med", y="y_med", color="red", marker="o", ax=ax)
    ax.set_title(f"{col} vs log1p(ingtrab)")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 9. Variables categóricas vs ingresos

In [ ]:
income_cats = [c for c in CATEGORICAL_RELEVANT["concentradohogar"] if c in income_df.columns]
cat_income_rows = []
for col in income_cats:
    tmp = income_df[[col, "ingtrab"]].dropna().copy()
    tmp[col] = tmp[col].astype("string")
    grouped = tmp.groupby(col)["ingtrab"].agg(n="count", mean="mean", median="median", p25=lambda x: x.quantile(.25), p75=lambda x: x.quantile(.75)).reset_index()
    grouped["variable"] = col
    cat_income_rows.append(grouped.rename(columns={col: "categoria"}))
cat_income = pd.concat(cat_income_rows, ignore_index=True)
display(cat_income.sort_values(["variable", "median"], ascending=[True, False]).round(2))

In [ ]:
for col in income_cats:
    tmp = income_df.copy()
    tmp[col] = tmp[col].astype("string")
    levels = tmp[col].value_counts().head(12).index
    tmp = tmp[tmp[col].isin(levels)].copy()
    tmp["_ing"] = tmp["ingtrab"].clip(upper=income_p99)
    fig, ax = plt.subplots(figsize=(10, 4.2))
    sns.boxplot(data=tmp, x=col, y="_ing", showfliers=False, color="#A0CBE8", ax=ax)
    ax.set_title(f"ingtrab por {col}")
    ax.set_ylabel("ingtrab")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 10. Estabilidad de relaciones por año

In [ ]:
per_year_corr_rows = []
for col in income_correlations.head(12)["variable"]:
    for year, g in income_df.groupby("anio"):
        x = pd.to_numeric(g[col], errors="coerce")
        y = g["ingtrab"]
        valid = x.notna() & y.notna()
        per_year_corr_rows.append({
            "variable": col,
            "anio": year,
            "Pearson": x[valid].corr(y[valid]),
            "Spearman": x[valid].rank().corr(y[valid].rank()),
            "n": int(valid.sum()),
        })
per_year_corr = pd.DataFrame(per_year_corr_rows)
display(per_year_corr.pivot(index="variable", columns="anio", values="Spearman").round(3))

In [ ]:
cat_year_income = []
for col in income_cats:
    tmp = income_df[[col, "anio", "ingtrab"]].dropna().copy()
    tmp[col] = tmp[col].astype("string")
    top = tmp[col].value_counts().head(8).index
    tmp = tmp[tmp[col].isin(top)]
    grouped = tmp.groupby(["anio", col])["ingtrab"].median().reset_index()
    grouped["variable"] = col
    cat_year_income.append(grouped.rename(columns={col: "categoria"}))
cat_year_income = pd.concat(cat_year_income, ignore_index=True)
display(cat_year_income.head(80).round(2))

## 11. Correlaciones generales y redundancias

In [ ]:
correlation_blocks = {
    "concentradohogar": ["ing_cor", "ingtrab", "trabajo", "sueldos", "negocio", "gasto_mon", "tot_integ", "ocupados", "percep_ing", "edad_jefe"],
    "poblacion": ["edad", "grado", "gradoaprob", "num_trabaj", "hijos_viv", "hijos_mue", "hijos_sob"],
    "viviendas": ["renta", "num_cuarto", "cuart_dorm", "tot_resid", "tot_hom", "tot_muj", "tot_hog"],
}
for table, cols in correlation_blocks.items():
    cols = [c for c in cols if c in clean_tables[table].columns]
    corr = clean_tables[table][cols].apply(pd.to_numeric, errors="coerce").corr()
    display(corr.round(2))
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
    ax.set_title(f"Correlaciones generales: {table}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 12. Pairplot

In [ ]:
pair_vars = ["ingtrab", "log1p_ingtrab", "ing_cor", "gasto_mon", "sueldos", "ocupados", "tot_integ", "edad_jefe"]
pair_vars = [c for c in pair_vars if c in income_df.columns]
pair_data = income_df[pair_vars].apply(pd.to_numeric, errors="coerce").dropna()
sample = pair_data.sample(n=min(3000, len(pair_data)), random_state=42)
g = sns.pairplot(sample, corner=True, diag_kind="hist", plot_kws={"alpha": .25, "s": 10, "edgecolor": "none"})
g.fig.suptitle("Pairplot de variables seleccionadas", y=1.02)
plt.show()
plt.close(g.fig)